# Train and backtest equities and synthetic options: $1T, $100B, $10B

This notebook explains and runs the current warehouse-streaming workflow. Change the universe in **Step 1**; the same trainer, model, document rules, and backtests serve all three thresholds.

**Flow:** configuration → fresh warehouse discovery → annual equity/option documents on demand → GPU optimization → predictions without warmup → four separate backtest books → coverage and results.

The saved notebook opens in **review** mode, so Run All displays existing artifacts without starting another training job. To train, set `MODE = "train"` and choose one or more `UNIVERSES`. Runs execute sequentially. Review reads saved outputs only; fresh training never consumes them as a corpus.

Requirements: a kernel with this repository's dependencies, access to the populated `quant-warehouse` store, and CUDA for training. Follow the repository environment instructions; `quant-warehouse` must come from `quantarb/quant-warehouse` on `main`. This notebook does not download data or install packages.

## Kernel dependency setup

This notebook needs the current `quant-warehouse` package layout, including
`quant_warehouse.platforms`. An older build can have the same version number
but lack that module. The cell below prints the active Python and package paths.
If that API is missing, it reinstalls the package from the repository's `main`
branch using **this kernel's Python**, preserving other installed dependencies.

After a repair, **restart the kernel and Run All** so previously imported modules
are cleared. An already compatible environment is left unchanged. If the install
fails, its pip error is shown directly (for example, missing GitHub access).

The check tests the full import in a fresh Python process as well as the running
kernel. If the installed files work but the kernel retains old modules, it asks
for **Kernel → Restart Kernel and Run All Cells**, without reinstalling again.
Rerunning a cell or refreshing the browser does not restart Python.

In [1]:
import sys
import subprocess
import importlib.util

print("Kernel Python:", sys.executable)
warehouse_spec = importlib.util.find_spec("quant_warehouse")
print("quant_warehouse:", warehouse_spec.origin if warehouse_spec else "not installed")
api_probe = (
    "from quant_warehouse.warehouse.backend import FrameFormat, StorageBackend; "
    "from quant_warehouse.platforms.data_providers.thetadata.options "
    "import read_thetadata_eod_option_chain"
)
# A separate process checks files on disk without the kernel's cached modules.
probe = subprocess.run([sys.executable, "-c", api_probe], capture_output=True, text=True)
if probe.returncode:
    print("Repairing quant-warehouse in this kernel's environment...", flush=True)
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--force-reinstall", "--no-deps",
        "quant-warehouse @ git+https://github.com/quantarb/quant-warehouse.git@main",
    ])
    probe = subprocess.run([sys.executable, "-c", api_probe], capture_output=True, text=True)
    if probe.returncode:
        raise RuntimeError("Warehouse import still fails in a fresh process:\n" + probe.stderr)
try:
    from quant_warehouse.warehouse.backend import FrameFormat, StorageBackend
    from quant_warehouse.platforms.data_providers.thetadata.options import read_thetadata_eod_option_chain
except ImportError:
    raise RuntimeError(
        "The installed package passes in a fresh process, but this kernel still has old "
        "quant_warehouse modules in memory. In Jupyter choose Kernel > Restart Kernel "
        "and Run All Cells. Rerunning this cell or refreshing the browser does not "
        "restart Python. No further package installation is needed."
    ) from None
print("Warehouse option-chain API is available.")

Kernel Python: /home/jlee153232/miniconda3/envs/quant-orchestrator/bin/python
quant_warehouse: /home/jlee153232/miniconda3/envs/quant-orchestrator/lib/python3.12/site-packages/quant_warehouse/__init__.py


Warehouse option-chain API is available.


## Step 1 — Choose the universe and dates

`min_market_cap` is an underlying-symbol filter, in USD. The loader queries stored US NYSE/NASDAQ profiles with ETF/fund exclusion flags, then requires at least two pre-cutoff equity prices. It discovers stored option dates independently for each selected share class. The resulting roster depends on warehouse metadata; it is **not** a reconstructed historical market-cap universe for each year.

Training uses calendar years before January 1, 2024. Scoring covers 2024 through September 9, 2026. Keep these dates fixed when comparing universes. Select `["1T"]`, `["100B"]`, `["10B"]`, or several thresholds. Each training launch gets a new output directory.

In [2]:
from pathlib import Path
from datetime import datetime, timezone
import importlib.util
import json
import os
import shlex
import subprocess
import sys
import uuid
import pandas as pd
from IPython.display import display

MODE = "review"                         # "train" launches fresh training + backtests
UNIVERSES = ["1T", "100B", "10B"]       # use ["1T"] for the smallest run
MARKET_CAPS = {"1T": 1_000_000_000_000, "100B": 100_000_000_000, "10B": 10_000_000_000}
EPOCHS = 1
BATCH_SIZE = 16
TRAIN_END = "2024-01-01"                 # exclusive training cutoff; must be January 1
PREDICTION_START = "2024-01-01"
PREDICTION_END = "2026-09-09"
# Optional exact output paths for review; otherwise select the latest warehouse-stream run.
REVIEW_RUNS = {}

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "scripts/train_multirate_mtl.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the quant-orchestrator checkout.")
EXPERIMENT = ROOT / "artifacts/multirate_recovery"
if MODE not in {"review", "train"} or not UNIVERSES or len(set(UNIVERSES)) != len(UNIVERSES):
    raise ValueError("Choose train/review and a nonempty list of distinct universes.")
if set(UNIVERSES) - MARKET_CAPS.keys():
    raise ValueError("Supported universes: 1T, 100B, 10B")
if EPOCHS < 1 or BATCH_SIZE < 1:
    raise ValueError("Epochs and batch size must be positive.")
cutoff, start, end = map(datetime.fromisoformat, [TRAIN_END, PREDICTION_START, PREDICTION_END])
if (cutoff.month, cutoff.day) != (1, 1) or not cutoff <= start <= end:
    raise ValueError("Use a January 1 cutoff <= prediction start <= prediction end.")
display(pd.DataFrame([{"universe": u, "min_market_cap_USD": MARKET_CAPS[u],
                       "mode": MODE, "epochs": EPOCHS} for u in UNIVERSES]))

,universe,min_market_cap_USD,mode,epochs
0,1T,1000000000000,review,1
1,100B,100000000000,review,1
2,10B,10000000000,review,1


## Option EDA — Contracts per symbol on the first trading day

Choose a year and market-cap threshold below. This cell queries the warehouse's
current profile universe and reads each symbol's **full stored option chain** on
that year's first NYSE trading session. It counts distinct contract identifiers
across **all strikes, expirations, calls, and puts**, before synthetic-basket
selection or quote-quality filtering. It does not build a training corpus.

The summary reports the average contracts per symbol **among symbols with a
stored chain**, plus the number of symbols missing that day's chain. Missing
chains are shown as missing, not treated as evidence of zero listed contracts.
The per-symbol table includes call and put counts. This EDA uses the profile
filter directly, without the trainer's additional pre-cutoff equity-price gate.
Change `EDA_YEAR` to compare another year; no later-date fallback is used.

The cell first previews five stored ThetaData option rows from the first symbol
with data, showing **every column without column truncation**. Change
`PREVIEW_ROWS` for a larger sample. These are warehouse-stored ThetaData records,
including normalized and derived fields, before the EDA expiration filter.

In [3]:
import exchange_calendars as xcals
import polars as pl
from quant_warehouse import Warehouse
from quant_warehouse.platforms.data_providers.thetadata.options import read_thetadata_eod_option_chain

EDA_UNIVERSE = UNIVERSES[0]  # "1T", "100B", or "10B"
EDA_YEAR = 2024
PREVIEW_ROWS = 5

warehouse = Warehouse()
calendar = xcals.get_calendar("XNYS", start=f"{EDA_YEAR - 1}-12-01", end=f"{EDA_YEAR}-12-31")
first_session = calendar.sessions_in_range(f"{EDA_YEAR}-01-01", f"{EDA_YEAR}-01-10")[0]
first_day = first_session.to_pydatetime().replace(tzinfo=None)
profiles = warehouse.catalog.query_symbol_profiles(
    provider="fmp", min_market_cap=MARKET_CAPS[EDA_UNIVERSE], country="US",
    exchanges=["NASDAQ", "NYSE"], exclude_etf=True, exclude_fund=True)
symbols = sorted({profile.symbol for profile in profiles})
contract_rows = []
option_data_preview = None
for symbol in symbols:
    chain = read_thetadata_eod_option_chain(
        symbol, start_date=first_day, end_date=first_day, backend=warehouse.backend)
    if chain.is_empty():
        contract_rows.append({"symbol": symbol, "contracts": None, "calls": None,
                              "puts": None, "status": "missing stored chain"})
        continue
    if option_data_preview is None:
        option_data_preview = chain.sort("contract_symbol").head(PREVIEW_ROWS).to_pandas()
        print(f"ThetaData stored option data: {symbol}, {first_day.date()}, "
              f"{chain.height:,} rows, {chain.width} columns")
        print("All columns:", ", ".join(chain.columns))
        with pd.option_context("display.max_columns", None, "display.max_colwidth", None,
                               "display.width", None, "display.max_rows", None):
            display(option_data_preview)
    contracts = chain.filter(pl.col("contract_symbol").is_not_null()).unique("contract_symbol")
    contract_rows.append({"symbol": symbol, "contracts": contracts.height,
        "calls": contracts.filter(pl.col("option_type") == "call").height,
        "puts": contracts.filter(pl.col("option_type") == "put").height,
        "status": "stored chain"})

if option_data_preview is None:
    print("No stored option data is available for this universe and date.")

option_contract_counts = pd.DataFrame(contract_rows, columns=["symbol", "contracts", "calls", "puts", "status"])
for column in ["contracts", "calls", "puts"]:
    option_contract_counts[column] = option_contract_counts[column].astype("Int64")
observed = option_contract_counts.loc[option_contract_counts["status"] == "stored chain", "contracts"]
option_contract_summary = pd.DataFrame([{
    "universe": EDA_UNIVERSE, "first_trading_day": first_day.date().isoformat(),
    "selected_symbols": len(symbols), "symbols_with_chain": len(observed),
    "symbols_missing_chain": len(symbols) - len(observed),
    "average_contracts_per_observed_symbol": observed.mean(),
    "median_contracts_per_observed_symbol": observed.median() if len(observed) else None,
    "min_contracts": observed.min(), "max_contracts": observed.max(),
}])
display(option_contract_summary)
display(option_contract_counts.sort_values(["contracts", "symbol"], ascending=[False, True], na_position="last"))

ThetaData stored option data: AAPL, 2024-01-02, 3,792 rows, 43 columns
All columns: date, quote_timestamp, underlying_symbol, expiration, strike, option_type, created_at, last_trade_time, open_price, high_price, low_price, last_trade_price, volume, count, bid_size, bid_exchange, bid, bid_condition, ask_size, ask_exchange, ask, ask_condition, snapshot_date, contract_symbol, data_interval, underlying_price, eod_date, dte, contract_size, theoretical_price, mark, close, close_size, change, change_percent, iv, delta, gamma, theta, vega, rho, mid, open_interest


,date,quote_timestamp,underlying_symbol,expiration,strike,option_type,created_at,last_trade_time,open_price,high_price,low_price,last_trade_price,volume,count,bid_size,bid_exchange,bid,bid_condition,ask_size,ask_exchange,ask,ask_condition,snapshot_date,contract_symbol,data_interval,underlying_price,eod_date,dte,contract_size,theoretical_price,mark,close,close_size,change,change_percent,iv,delta,gamma,theta,vega,rho,mid,open_interest
0,2024-01-02 00:00:00.004907,2024-01-02,AAPL,2024-01-05,50.0,call,2262-04-11,2262-04-11,0.0,0.0,0.0,0.0,0.0,NaN,60.0,7,134.75,NaN,15.0,47,135.9,NaN,2024-01-02,AAPL240105C00050000,eod,185.64,2024-01-02,3.0,100.0,135.325,135.325,0.0,0.0,0.0,NaN,0.0,1.0,0.0,0.0,0.0,0.0,135.325,NaN
1,2024-01-02 00:00:00.004907,2024-01-02,AAPL,2024-01-05,60.0,call,2262-04-11,2262-04-11,0.0,0.0,0.0,0.0,0.0,NaN,60.0,7,124.75,NaN,3.0,60,125.9,NaN,2024-01-02,AAPL240105C00060000,eod,185.64,2024-01-02,3.0,100.0,125.325,125.325,0.0,0.0,0.0,NaN,0.0,1.0,0.0,0.0,0.0,0.0,125.325,NaN
2,2024-01-02 00:00:00.004907,2024-01-02,AAPL,2024-01-05,65.0,call,2262-04-11,2262-04-11,0.0,0.0,0.0,0.0,0.0,NaN,60.0,7,119.75,NaN,5.0,60,120.9,NaN,2024-01-02,AAPL240105C00065000,eod,185.64,2024-01-02,3.0,100.0,120.325,120.325,0.0,0.0,0.0,NaN,0.0,1.0,0.0,0.0,0.0,0.0,120.325,NaN
3,2024-01-02 00:00:00.004907,2024-01-02,AAPL,2024-01-05,70.0,call,2262-04-11,2262-04-11,0.0,0.0,0.0,0.0,0.0,NaN,60.0,1,115.15,NaN,30.0,11,115.9,NaN,2024-01-02,AAPL240105C00070000,eod,185.64,2024-01-02,3.0,100.0,115.525,115.525,0.0,0.0,0.0,NaN,0.0,1.0,0.0,0.0,0.0,0.0,115.525,NaN
4,2024-01-02 00:00:00.004907,2024-01-02,AAPL,2024-01-05,75.0,call,2262-04-11,2262-04-11,0.0,0.0,0.0,0.0,0.0,NaN,30.0,11,109.90,NaN,12.0,11,110.9,NaN,2024-01-02,AAPL240105C00075000,eod,185.64,2024-01-02,3.0,100.0,110.400,110.400,0.0,0.0,0.0,NaN,0.0,1.0,0.0,0.0,0.0,0.0,110.400,NaN


,universe,first_trading_day,selected_symbols,symbols_with_chain,symbols_missing_chain,average_contracts_per_observed_symbol,median_contracts_per_observed_symbol,min_contracts,max_contracts
0,1T,2024-01-02,14,12,2,3300.75,3295.5,178,7704


,symbol,contracts,calls,puts,status
2,AVGO,7704,3852,3852,stored chain
11,NVDA,5492,2746,2746,stored chain
8,META,4098,2049,2049,stored chain
13,TSLA,3910,1955,1955,stored chain
0,AAPL,3792,1896,1896,stored chain
1,AMZN,3532,1766,1766,stored chain
9,MSFT,3059,1529,1530,stored chain
6,GOOGL,2357,1178,1179,stored chain
7,LLY,2236,1118,1118,stored chain
5,GOOG,1869,934,935,stored chain


## Option filtering — Keep expirations in the selected year

Keep contracts where **`expiration.year == EDA_YEAR`**. For the first trading
session of 2024, this keeps all 2024 expirations, regardless of when the contract
first appeared or traded. There is no origination-date condition.

The expiration-year table shows where the full first-day chain falls before
filtering. The per-symbol table compares counts before and after filtering.
Unknown expiration dates are excluded and counted separately. Averages include
symbols with stored chains, even if their retained count is zero; missing chains
remain missing. `option_expiration_audit` contains each candidate and its keep flag.

This EDA filter does not change the trainer or warehouse storage.

In [ ]:
expiration_audits = []
filtered_rows = []
for symbol in symbols:
    chain = read_thetadata_eod_option_chain(
        symbol, start_date=first_day, end_date=first_day, backend=warehouse.backend)
    if chain.is_empty():
        filtered_rows.append({"symbol": symbol, "status": "missing stored chain"})
        continue
    contracts = chain.filter(pl.col("contract_symbol").is_not_null()).unique("contract_symbol")
    annotated = contracts.with_columns(
        (pl.col("expiration").dt.year() == EDA_YEAR).fill_null(False).alias("keep_same_year"))
    expiration_audits.append(annotated.select(
        "underlying_symbol", "contract_symbol", "expiration", "strike", "option_type", "keep_same_year"))
    kept = annotated.filter(pl.col("keep_same_year"))
    unknown = annotated["expiration"].null_count()
    filtered_rows.append({
        "symbol": symbol, "status": "stored chain", "before": contracts.height,
        "kept_same_year": kept.height, "removed": contracts.height - kept.height,
        "removed_different_year": contracts.height - kept.height - unknown,
        "removed_unknown_expiration": unknown,
        "calls_kept": kept.filter(pl.col("option_type") == "call").height,
        "puts_kept": kept.filter(pl.col("option_type") == "put").height,
    })

count_columns = ["before", "kept_same_year", "removed", "removed_different_year",
                 "removed_unknown_expiration", "calls_kept", "puts_kept"]
same_year_contract_counts = pd.DataFrame(filtered_rows, columns=["symbol", "status", *count_columns])
for column in count_columns:
    same_year_contract_counts[column] = same_year_contract_counts[column].astype("Int64")
observed_same_year = same_year_contract_counts.loc[same_year_contract_counts["status"] == "stored chain"]
same_year_summary = pd.DataFrame([{
    "universe": EDA_UNIVERSE, "chain_day": first_day.date().isoformat(),
    "symbols_with_chain": len(observed_same_year),
    "symbols_missing_chain": len(symbols) - len(observed_same_year),
    "average_before": observed_same_year["before"].mean(),
    "average_after": observed_same_year["kept_same_year"].mean(),
    "total_kept": observed_same_year["kept_same_year"].sum(min_count=1),
    "total_removed": observed_same_year["removed"].sum(min_count=1),
}])
display(same_year_summary)
display(same_year_contract_counts.sort_values(
    ["kept_same_year", "symbol"], ascending=[False, True], na_position="last"))

option_expiration_audit = pl.concat(expiration_audits) if expiration_audits else pl.DataFrame()
if not option_expiration_audit.is_empty():
    expiration_year_counts = (option_expiration_audit
        .with_columns(pl.col("expiration").dt.year().alias("expiration_year"))
        .group_by("expiration_year").len(name="contracts").sort("expiration_year")
        .with_columns((100 * pl.col("contracts") / pl.col("contracts").sum()).alias("percent")))
    display(expiration_year_counts.to_pandas())


,universe,chain_day,symbols_with_chain,symbols_missing_chain,average_before,average_after,total_kept,total_removed
0,1T,2024-01-02,12,2,3300.75,2565.25,30783,8826


,symbol,status,before,kept_same_year,removed,removed_different_year,removed_unknown_expiration,calls_kept,puts_kept
2,AVGO,stored chain,7704,5940,1764,1764,0,2970,2970
11,NVDA,stored chain,5492,4056,1436,1436,0,2028,2028
13,TSLA,stored chain,3910,3114,796,796,0,1557,1557
8,META,stored chain,4098,3006,1092,1092,0,1503,1503
1,AMZN,stored chain,3532,2808,724,724,0,1404,1404
0,AAPL,stored chain,3792,2796,996,996,0,1398,1398
9,MSFT,stored chain,3059,2383,676,676,0,1191,1192
6,GOOGL,stored chain,2357,1989,368,368,0,994,995
7,LLY,stored chain,2236,1840,396,396,0,920,920
5,GOOG,stored chain,1869,1613,256,256,0,806,807


,expiration_year,contracts,percent
0,2024,30783,77.717185
1,2025,6840,17.268803
2,2026,1986,5.014012


## Option filtering — Finish in the money at expiration

From the same-year-expiration candidates, keep **calls with underlying spot above
strike** and **puts with underlying spot below strike** on the expiration trading
day. At-the-money contracts are excluded. This is an **in-the-money** filter,
not a net-profit test: an option can pay intrinsic value and still lose money
relative to its purchase premium.

Use the last NYSE session on or before expiration, including Friday for Saturday
expirations. The underlying mark is the median finite positive `underlying_price`
in that underlying's stored ThetaData chain **on that exact session**. This is
an EOD proxy, not an official contract-specific settlement fixing; no earlier
quote is carried forward. Forward stock splits between entry and expiration
adjust the strike basis. Reverse splits or invalid split ratios are marked
unknown rather than guessing an adjusted deliverable.

Missing expiration-day prices and unsupported terms are counted as **unknown**,
not as out-of-the-money. Only confirmed in-the-money contracts survive. This uses
future outcomes for retrospective EDA and does not alter training or become an
entry-time trading signal. `option_itm_audit` exposes each decision and its inputs.

In [ ]:
from quant_warehouse.platforms.data_providers.thetadata.options import read_option_chain_arctic

itm_audits = []
if not option_expiration_audit.is_empty():
    candidates = option_expiration_audit.filter(pl.col("keep_same_year"))
    for (symbol,), contracts in candidates.group_by("underlying_symbol", maintain_order=True):
        expirations = contracts["expiration"].unique().to_list()
        sessions = {expiry: calendar.date_to_session(
            expiry.date().isoformat(), direction="previous").to_pydatetime().replace(tzinfo=None)
            for expiry in expirations}
        history = read_option_chain_arctic(
            symbol, start_date=min(sessions.values()), end_date=max(sessions.values()),
            columns=["snapshot_date", "underlying_price"], backend=warehouse.backend)
        spots = {}
        if not history.is_empty() and "underlying_price" in history.columns:
            valid_spots = (history.filter(pl.col("underlying_price").is_finite() &
                                         (pl.col("underlying_price") > 0))
                           .group_by("snapshot_date").agg(pl.col("underlying_price").median().alias("spot")))
            spots = dict(valid_spots.select("snapshot_date", "spot").iter_rows())
        del history
        splits = warehouse.read_fundamentals(
            symbol, section="historical_splits", start=first_day.date().isoformat(),
            end=max(sessions.values()).date().isoformat())
        split_rows = splits.to_dicts() if not splits.is_empty() else []
        settlement_rows = []
        for expiry, session in sessions.items():
            factor, supported = 1.0, True
            for split in split_rows:
                if first_day < split["date"] <= session:
                    numerator, denominator = split.get("numerator"), split.get("denominator")
                    if numerator is None or denominator is None or not (numerator > 0 and denominator > 0):
                        supported = False
                        continue
                    ratio = numerator / denominator
                    if not (1 <= ratio < float("inf")):
                        supported = False
                    else:
                        factor *= ratio
            settlement_rows.append({"expiration": expiry, "expiration_session": session,
                "expiration_spot": spots.get(session), "split_factor": factor,
                "supported_split": supported})
        settlements = pl.DataFrame(settlement_rows).with_columns(
            pl.col("expiration").cast(contracts.schema["expiration"]),
            pl.col("expiration_spot").cast(pl.Float64))
        audit = contracts.join(settlements, on="expiration", how="left").with_columns(
            (pl.col("strike") / pl.col("split_factor")).alias("expiration_strike"))
        known = (pl.col("expiration_spot").is_not_null() & pl.col("supported_split") &
                 pl.col("strike").is_finite() & pl.col("option_type").is_in(["call", "put"]))
        intrinsic = (pl.when(pl.col("option_type") == "call")
                     .then(pl.col("expiration_spot") - pl.col("expiration_strike"))
                     .otherwise(pl.col("expiration_strike") - pl.col("expiration_spot"))).clip(lower_bound=0)
        audit = audit.with_columns(
            pl.when(known).then(intrinsic).otherwise(None).alias("intrinsic_per_adjusted_share"))
        audit = audit.with_columns(
            pl.when(~pl.col("supported_split")).then(pl.lit("unknown_split_adjustment"))
            .when(pl.col("intrinsic_per_adjusted_share").is_null()).then(pl.lit("unknown_expiration_data"))
            .when(pl.col("intrinsic_per_adjusted_share") > 0).then(pl.lit("in_the_money"))
            .otherwise(pl.lit("out_or_at_the_money")).alias("expiry_status"))
        itm_audits.append(audit)

option_itm_audit = pl.concat(itm_audits, how="diagonal_relaxed") if itm_audits else pl.DataFrame()
if option_itm_audit.is_empty():
    print("No same-year candidates to evaluate; run the preceding EDA cells first.")
else:
    per_symbol_itm = option_itm_audit.group_by("underlying_symbol").agg(
        pl.len().alias("before_itm_filter"),
        (pl.col("expiry_status") == "in_the_money").sum().alias("kept_itm"),
        (pl.col("expiry_status") == "out_or_at_the_money").sum().alias("removed_otm_or_atm"),
        pl.col("expiry_status").str.starts_with("unknown").sum().alias("unknown"))
    # Include symbols with an observed chain but no same-year candidates as zeros.
    base_symbols = same_year_contract_counts.loc[
        same_year_contract_counts["status"] == "stored chain", ["symbol"]].rename(columns={"symbol": "underlying_symbol"})
    itm_counts = base_symbols.merge(per_symbol_itm.to_pandas(), on="underlying_symbol", how="left").fillna(0)
    count_fields = ["before_itm_filter", "kept_itm", "removed_otm_or_atm", "unknown"]
    itm_counts[count_fields] = itm_counts[count_fields].astype(int)
    totals = itm_counts[count_fields].sum().to_dict()
    display(pd.DataFrame([{"universe": EDA_UNIVERSE, "year": EDA_YEAR, **totals,
                           "average_itm_per_observed_symbol": itm_counts["kept_itm"].mean()}]))
    display(itm_counts.sort_values("kept_itm", ascending=False))
    display(option_itm_audit.select("underlying_symbol", "contract_symbol", "expiration_session",
        "strike", "split_factor", "expiration_strike", "expiration_spot", "expiry_status").head(20).to_pandas())
    itm_options = option_itm_audit.filter(pl.col("expiry_status") == "in_the_money")
    print("itm_options contains ITM contracts; purchase premiums were not deducted.")

,universe,year,before_itm_filter,kept_itm,removed_otm_or_atm,unknown,average_itm_per_observed_symbol
0,1T,2024,30783,15390,15393,0,1282.5


,underlying_symbol,before_itm_filter,kept_itm,removed_otm_or_atm,unknown
2,AVGO,5940,2970,2970,0
9,NVDA,4056,2028,2028,0
11,TSLA,3114,1557,1557,0
6,META,3006,1503,1503,0
1,AMZN,2808,1404,1404,0
0,AAPL,2796,1396,1400,0
7,MSFT,2383,1192,1191,0
4,GOOGL,1989,993,996,0
5,LLY,1840,920,920,0
3,GOOG,1613,808,805,0


,underlying_symbol,contract_symbol,expiration_session,strike,split_factor,expiration_strike,expiration_spot,expiry_status
0,AAPL,AAPL240216P00200000,2024-02-16,200.0,1.0,200.0,182.31,in_the_money
1,AAPL,AAPL_call_20240119_170,2024-01-19,170.0,1.0,170.0,191.56,in_the_money
2,AAPL,AAPL240920P00255000,2024-09-20,255.0,1.0,255.0,228.20,in_the_money
3,AAPL,AAPL_call_20240216_300,2024-02-16,300.0,1.0,300.0,182.31,out_or_at_the_money
4,AAPL,AAPL_call_20240315_120,2024-03-15,120.0,1.0,120.0,172.62,in_the_money
5,AAPL,AAPL240202C00120000,2024-02-02,120.0,1.0,120.0,185.85,in_the_money
6,AAPL,AAPL_call_20240126_200,2024-01-26,200.0,1.0,200.0,192.42,out_or_at_the_money
7,AAPL,AAPL240315P00105000,2024-03-15,105.0,1.0,105.0,172.62,out_or_at_the_money
8,AAPL,AAPL_call_20240112_70,2024-01-12,70.0,1.0,70.0,185.92,in_the_money
9,AAPL,AAPL_put_20240126_200,2024-01-26,200.0,1.0,200.0,192.42,in_the_money


itm_options contains ITM contracts; purchase premiums were not deducted.


## Step 2 — Understand what is loaded, and when

1. Discover equity profiles and price histories for the requested threshold, and option-date coverage for every eligible symbol.
2. Load issuer financial families, sparse events, macro series, and peer context from the warehouse. Peer context needs a cross-universe source pass on first use, so startup grows with universe size.
3. Assemble annual tensors only as training requests documents. The source cache is bounded and one CPU batch is prefetched while the GPU works.

There is no corpus-building command, prior-run roster, feature export, or fitted normalization pass. A packaged **field-name schema** defines the architecture; it contains no training examples. Numeric values use the fixed transform `sign(x) * log1p(abs(x)) / 10`, with missingness preserved.

“On demand” does not mean zero preparation: discovery, source reads, peer context, and the first batch precede the first optimizer update. Recorded first updates were 27.46 seconds for the completed $1T run and 123.11 seconds for the later $100B attempt. These are measurements of those runs, not promises for $10B.

## Step 3 — Build the synthetic option price series

On the **actual first NYSE trading session of each year**, select up to five positive-DTE expirations per right across the available range. With fewer than five expirations, use all available ones. Each selected call or put expiration becomes a basket of **all its strikes**, initially equal weighted.

The basket's membership is frozen for that year. Its time series follows those selected contracts; later contracts are not added, and there is no roll. These are historical option-price series, not theoretical prices inferred from equities. Basket terms and the underlying issuer/equity context accompany the option's own price stream.

- Missing first-session chains are recorded and that year is skipped, without selecting a later chain.
- A daily basket quote requires every constituent; missing members are not renormalized away.
- Forward splits adjust strikes and contract counts to preserve economic exposure.
- Expiry uses split-consistent intrinsic value from the underlying spot in the option chain. Missing required settlement values fail backtesting.
- A basket with no complete quote fails the run. Reverse-split deliverables require an explicit mapping.

Coverage can differ by year even when an underlying has stored options history. The report below exposes these differences.

## Step 4 — From fields to subtokens, tokens, and documents

| Level | Meaning in this run |
|---|---|
| Raw feature | An observed price, financial field, basket term, or sparse event value, with its date and missingness |
| Family subtoken | A learned vector encoding one feature family's observed fields; the family is **not averaged into one scalar** |
| Token | A learned representation combining family information and available time-aligned context |
| Annual document | One instrument's native observations within a calendar year, plus a memory-prefix position |
| Annual memory | Detached learned state passed from an earlier document to a later year of the **same instrument** |

For an equity, 2024 covers January 1–December 31, and 2025 starts January 1 with the state left by 2024. January 1 need not be a trading day. Daily, quarterly, annual, and sparse streams keep their native observation dates inside that annual document; quarterly financial observations do not create separate quarterly training documents.

Every newly formed annual option basket has its own identity, so its state does **not** carry into next year's newly selected basket. Equities retain their identity across years. Instruments are interleaved for batching while preserving each instrument's chronological order. Gradients do not backpropagate across years through detached memory.

An epoch visits all generated documents. Thus equity count is not sample count: historical years contribute equity documents, and each underlying/year can contribute up to ten option documents. Sparse option history can yield short documents; the final coverage gate requires temporal option documents for every eligible option underlying.

## Step 5 — What the optimizer learns

The run uses a 64-dimensional, two-layer multi-rate Transformer with four attention heads, FP32 CUDA, AdamW, batch size 16, and seed 0. Document-level tasks and MRL are disabled in this workflow.

Supervised Oracle/HITS targets come from each instrument's own price path; equity documents can also use warehouse event targets. Classification supervision is restricted to actual labeled events: missing labels are masked, not converted into “no event.” Option documents contribute their own targets.

Both next-token prediction and masked reconstruction train on observed feature sequences, with reconstruction weight 0.1. Each batch performs forward evaluation, weighted loss calculation, backpropagation, gradient clipping, and an optimizer update. Finite-loss/gradient checks stop invalid training. Checkpoints record model, optimizer, annual memory, normalization, and objective-observation counts.

Current warehouse-stream checkpoints are saved for audit; this loader does not implement resume or standalone inference from them. A fresh launch starts new weights. Backtests run inside the same process after each completed epoch.

In [5]:
def training_command(universe, output):
    return [sys.executable, str(ROOT / "scripts/train_multirate_mtl.py"),
        "--min-market-cap", str(MARKET_CAPS[universe]), "--output-dir", str(output),
        "--epochs", str(EPOCHS), "--batch-size", str(BATCH_SIZE),
        "--d-model", "64", "--num-heads", "4", "--layers", "2",
        "--mrl-dimensions", "", "--disable-document-tasks", "--device", "cuda",
        "--warehouse-start-date", "1900-01-01", "--train-end-date", TRAIN_END,
        "--prediction-start-date", PREDICTION_START, "--prediction-end-date", PREDICTION_END,
        "--checkpoint-every-batches", "10", "--progress-every-batches", "1",
        "--self-supervision", "both", "--reconstruction-weight", "0.1",
        "--sequence-mode", "annual_memory", "--skip-embeddings", "--skip-t-sne"]

# Preview is read-only. It does not query data, create a corpus, or launch training.
for universe in UNIVERSES:
    print(universe, shlex.join(training_command(universe, EXPERIMENT / universe / "NEW_RUN")), "\n")

1T /home/jlee153232/miniconda3/envs/quant-orchestrator/bin/python /home/jlee153232/PycharmProjects/quant-orchestrator/scripts/train_multirate_mtl.py --min-market-cap 1000000000000 --output-dir /home/jlee153232/PycharmProjects/quant-orchestrator/artifacts/multirate_recovery/1T/NEW_RUN --epochs 1 --batch-size 16 --d-model 64 --num-heads 4 --layers 2 --mrl-dimensions '' --disable-document-tasks --device cuda --warehouse-start-date 1900-01-01 --train-end-date 2024-01-01 --prediction-start-date 2024-01-01 --prediction-end-date 2026-09-09 --checkpoint-every-batches 10 --progress-every-batches 1 --self-supervision both --reconstruction-weight 0.1 --sequence-mode annual_memory --skip-embeddings --skip-t-sne 

100B /home/jlee153232/miniconda3/envs/quant-orchestrator/bin/python /home/jlee153232/PycharmProjects/quant-orchestrator/scripts/train_multirate_mtl.py --min-market-cap 100000000000 --output-dir /home/jlee153232/PycharmProjects/quant-orchestrator/artifacts/multirate_recovery/100B/NEW_RUN -

## Step 6 — Run training, or select existing outputs

In `train` mode this cell runs the existing trainer as a subprocess using the notebook kernel's Python, streams its logs, and waits for training **and** backtests. Each universe runs sequentially; a failed run stops the sequence with the traceback. Interrupting this cell terminates only the process it launched. Avoid launching another copy on the same GPU while this cell is active.

In `review` mode this cell selects existing warehouse-stream outputs without loading any market data. It prefers the latest run, even if failed, so failures are visible rather than hidden by an older success. Set `REVIEW_RUNS` to inspect a particular run. Re-running the training cell creates fresh runs; it never overwrites or resumes a prior run.

In [6]:
def read_json(path):
    return json.loads(path.read_text()) if path.is_file() else {}

RUNS = {}
if MODE == "train":
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError("Select a CUDA-enabled kernel before training.")
    if importlib.util.find_spec("quant_warehouse") is None:
        raise RuntimeError("Install the repository's quant-warehouse dependency in this kernel.")
    env = os.environ.copy()
    env.update(POLARS_MAX_THREADS="8", OMP_NUM_THREADS="4", PYTHONUNBUFFERED="1")
    env["PYTHONPATH"] = str(ROOT) + (os.pathsep + env["PYTHONPATH"] if env.get("PYTHONPATH") else "")
    for universe in UNIVERSES:
        name = "warehouse_stream_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
        output = EXPERIMENT / universe / name
        output.parent.mkdir(parents=True, exist_ok=True)  # Trainer creates the run directory.
        RUNS[universe] = output
        command = training_command(universe, output)
        print(f"Starting {universe}: {output}", flush=True)
        with Path(str(output) + ".log").open("w") as log:
            process = subprocess.Popen(command, cwd=ROOT, env=env, stdout=subprocess.PIPE,
                                       stderr=subprocess.STDOUT, text=True, bufsize=1)
            try:
                for line in process.stdout:
                    log.write(line); log.flush()
                    print(line, end="", flush=True)
                returncode = process.wait()
            except BaseException:
                process.terminate()
                try:
                    process.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    process.kill(); process.wait()
                raise
        if returncode:
            failure = output / "failure.txt"
            raise RuntimeError(f"Run failed: {output}\n" +
                               (failure.read_text() if failure.exists() else f"See {output}.log"))
        if read_json(output / "status.json").get("stage") != "complete":
            raise RuntimeError(f"Process exited without a complete status: {output}")
else:
    for universe in UNIVERSES:
        if universe in REVIEW_RUNS:
            candidate = Path(REVIEW_RUNS[universe]).expanduser()
            output = candidate if candidate.is_absolute() else ROOT / candidate
            if not (output / "configuration.json").is_file():
                raise ValueError(f"Not a run directory: {output}")
        else:
            candidates = sorted((EXPERIMENT / universe).glob("warehouse_stream_*/configuration.json"))
            candidates = [p.parent for p in candidates
                          if read_json(p).get("dataset_mode") == "warehouse_on_demand"]
            output = candidates[-1] if candidates else None
        if output is not None:
            RUNS[universe] = output
        else:
            print(f"{universe}: no warehouse-stream run found; choose train mode to create one.")

display(pd.DataFrame([{"universe": u, "run": str(p),
                       "stage": read_json(p / "status.json").get("stage", "status unavailable")}
                      for u, p in RUNS.items()]))

10B: no warehouse-stream run found; choose train mode to create one.


,universe,run,stage
0,1T,/home/jlee153232/PycharmProjects/quant-orchest...,complete
1,100B,/home/jlee153232/PycharmProjects/quant-orchest...,failed


## Step 7 — Confirm actual equity and option coverage

Read the run's **own** configuration, universe, and coverage outputs. `trained` counts reflect the last coverage write and can lag the live status by a few batches. Counts accumulate across epochs. An underlying listed in the selected universe is not proof that its option documents trained; check actual observation counts and the final run status.

The completed $1T run included 13 equities and 11 option underlyings. Berkshire's two share classes had no stored option series. The later $100B attempt reached batch 101 before stopping on `CVS/2021` because the four-day put basket had no complete quote. No success or $10B timing is inferred from those attempts.

In [7]:
coverage_rows, option_rows, cohort_rows = [], [], []
for universe, output in RUNS.items():
    config = read_json(output / "configuration.json")
    roster = read_json(output / "universe.json")
    coverage = read_json(output / "option_coverage.json")
    status = read_json(output / "status.json")
    startup = read_json(output / "startup_timing.json")
    if config.get("min_market_cap") != MARKET_CAPS[universe]:
        raise ValueError(f"Universe/path mismatch: {output}")
    trained = coverage.get("trained", {})
    coverage_rows.append({"universe": universe, "stage": status.get("stage"),
        "equities": len(roster.get("equity_symbols", [])),
        "expected_option_underlyings": len(roster.get("option_symbols", [])),
        "observed_option_underlyings": len(trained),
        "option_documents": sum(r.get("documents", 0) for r in trained.values()),
        "option_price_observations": sum(r.get("price_observations", 0) for r in trained.values()),
        "first_update_seconds": startup.get("first_optimizer_update_seconds"),
        "corpus_built": startup.get("corpus_built")})
    for symbol, row in trained.items():
        option_rows.append({"universe": universe, "symbol": symbol, **row})
    for symbol, detail in coverage.get("coverage", {}).items():
        for row in detail.get("cohorts", []):
            cohort_rows.append({"universe": universe, "symbol": symbol, **row})
    missing = sorted(set(roster.get("equity_symbols", [])) - set(roster.get("option_symbols", [])))
    print(f"{universe}: no eligible stored pre-cutoff options: {missing}")
    if (output / "failure.txt").exists():
        print(f"{universe} failure: " + (output / "failure.txt").read_text().strip().splitlines()[-1])
coverage_summary = pd.DataFrame(coverage_rows)
option_coverage = pd.DataFrame(option_rows)
cohorts = pd.DataFrame(cohort_rows)
display(coverage_summary)
if not option_coverage.empty:
    display(option_coverage.groupby("universe")[["documents", "price_observations", "temporal_documents"]].sum())
if not cohorts.empty:
    display(cohorts.groupby(["universe", "status"]).size().rename("underlying_years").to_frame())
# Inspect option_coverage or cohorts directly for every symbol/year; no truncated source data is used in training.

1T: no eligible stored pre-cutoff options: ['BRK-A', 'BRK-B']
100B: no eligible stored pre-cutoff options: ['ABALX', 'BALFX', 'BNY', 'BRK-A', 'BRK-B', 'FNPFX', 'RICAX', 'RICBX', 'SOJE', 'TBB', 'TBC', 'VZA']
100B failure: ValueError: CVS/2021: frozen baskets without any complete quote: ['OPT_CVS_2021_PUT_DTE_4']


,universe,stage,equities,expected_option_underlyings,observed_option_underlyings,option_documents,option_price_observations,first_update_seconds,corpus_built
0,1T,complete,13,11,11,790,51218,27.460330,False
1,100B,failed,127,115,21,895,42408,123.105659,False


,documents,price_observations,temporal_documents
universe,,,
100B,895,42408,762
1T,790,51218,744


underlying_years
universe status                                       
100B     loaded                                     95
         missing_first_session_chain                53
1T       loaded                                    112
         missing_first_session_chain                36

## Step 8 — Score the held-out years without warmup

After each epoch, inference starts with empty annual memory at the requested start date. It does not replay pre-cutoff history. Memory can carry between requested years of the same equity. Only dates with observed instrument prices become trading scores; issuer/macro context alone cannot create a tradeable day.

The trainer checks duplicate predictions, omitted priced dates, and nonfinite scores. Predictions and prices are then saved as **backtest outputs**, never reused as a fresh training corpus.

Financial features retain warehouse-recorded observation dates. No additional reporting lag or historical data-vintage reconstruction is applied. The selected universe also comes from stored profiles. These are research timing/universe conventions, not a claim of a fully reconstructed point-in-time investable universe.

## Step 9 — Understand the four backtest books

| Book | Decisions and exposure |
|---|---|
| Equity long | Existing HITS strategy, long equity positions |
| Equity short | Existing HITS strategy, short equity positions |
| Long calls | Bullish **equity** signals buy the year's frozen call baskets |
| Long puts | Bearish **equity** signals buy the year's frozen put baskets |

Each year/book starts from its own $100,000. Returns are not additive across books. Options use equity signals as the decision maker even though the model also scores option instruments. Option orders execute no earlier than the next equity session and require complete basket quotes. The capacity planner holds positions awaiting an executable exit; expiry forces settlement without rolling.

The shared fixed-weight return engine applies 5.5 bps per unit of turnover; options also incur modeled bid/ask spread costs. Missing option marks carry forward **for valuation only**, and reports count stale position-days and open positions at period end. This is not an exact broker cash/margin ledger.

The table below uses `capital_return` (final equity / starting cash − 1), including the initial day's capital effect. It does not substitute the equity report's differently based `total_return`. A missing report is shown as missing, never zero.

In [8]:
report_rows, timing_rows = [], []
for universe, output in RUNS.items():
    files = sorted((output / "epoch_validation").glob("epoch_*/results.json"))
    if not files:
        print(f"{universe}: no completed epoch backtests in {output.name}")
    for path in files:
        epoch = int(path.parent.name.split("_")[-1])
        reports = json.loads(path.read_text())
        for report in reports:
            row = {"universe": universe, "epoch": epoch, **report}
            row["book"] = ("equity_" + report["side"] if report["asset_class"] == "equity" else report["side"])
            report_rows.append(row)
        timing_rows.append({"universe": universe, "epoch": epoch, **read_json(path.parent / "timing.json")})
results = pd.DataFrame(report_rows)
if not results.empty:
    returns = results.pivot(index=["universe", "epoch", "year"], columns="book", values="capital_return")
    returns = returns.reindex(columns=["equity_long", "equity_short", "long_calls", "long_puts"])
    display(returns.style.format("{:+.2%}", na_rep="missing"))
    detail_columns = ["universe", "epoch", "year", "book", "sharpe", "max_drawdown", "entries",
                      "stale_valuation_position_days", "open_positions_at_period_end"]
    display(results.reindex(columns=detail_columns))
if timing_rows:
    display(pd.DataFrame(timing_rows))

100B: no completed epoch backtests in warehouse_stream_20260913T032457Z


,universe,epoch,year,book,sharpe,max_drawdown,entries,stale_valuation_position_days,open_positions_at_period_end
0,1T,1,2024,equity_long,1.343739,-0.114361,169,NaN,NaN
1,1T,1,2024,equity_short,-2.477028,-0.200859,163,NaN,NaN
2,1T,1,2024,long_calls,0.485548,-0.345860,302,196.0,9.0
3,1T,1,2024,long_puts,-2.724612,-0.828264,305,577.0,6.0
4,1T,1,2025,equity_long,1.126882,-0.129504,207,NaN,NaN
5,1T,1,2025,equity_short,-1.415826,-0.201314,195,NaN,NaN
6,1T,1,2025,long_calls,0.179244,-0.526800,359,32.0,5.0
7,1T,1,2025,long_puts,-1.659229,-0.552280,336,59.0,7.0
8,1T,1,2026,equity_long,1.353587,-0.082731,137,NaN,NaN
9,1T,1,2026,equity_short,-0.960276,-0.117429,126,NaN,NaN


,universe,epoch,inference_seconds,backtest_seconds,predictions,inference_initialization
0,1T,1,144.429703,1.742271,41139,empty_memory_no_warmup


## Step 10 — Locate checkpoints, failures, and detailed artifacts

A completed run must have a complete status, a final epoch checkpoint, both trained asset classes, and all four books for each requested year. The checks below apply to completed runs only. A failed run is retained for diagnosis and is not silently replaced with an older result.

`cohort_members/` contains frozen membership audit outputs. `epoch_validation/epoch_XXXX/` contains predictions, price snapshots, timings, and year/asset reports. The `.log` alongside the run records batch-level progress. Keep the run path to reproduce or inspect a specific result.

To compare another threshold, change Step 1 and rerun from there. The same feature, model, and backtest code applies at every threshold. Larger universes may expose warehouse gaps, as the $100B attempt did; choosing $10B is supported but is not evidence that a full $10B options run has already passed.

In [9]:
for universe, output in RUNS.items():
    status = read_json(output / "status.json")
    print(f"\n{universe}: {output}")
    print("Log:", Path(str(output) + ".log"))
    print("Coverage:", output / "option_coverage.json")
    print("Failure details:", output / "failure.txt" if (output / "failure.txt").exists() else "none recorded")
    if status.get("stage") != "complete":
        print("Not marked complete; no completed-run assertion is made.")
        continue
    config = read_json(output / "configuration.json")
    final_epoch = int(config["epochs"])
    assert (output / f"epoch_{final_epoch:04d}.pt").is_file()
    assert status["documents"].get("equity", 0) > 0 and status["documents"].get("option", 0) > 0
    roster = read_json(output / "universe.json")
    trained = read_json(output / "option_coverage.json")["trained"]
    assert set(roster["option_symbols"]) <= set(trained)
    assert all(trained[s]["temporal_documents"] > 0 for s in roster["option_symbols"])
    years = range(int(config["prediction_start_date"][:4]), int(config["prediction_end_date"][:4]) + 1)
    expected = {(year, asset, side) for year in years for asset, sides in
                [("equity", ("long", "short")), ("option", ("long_calls", "long_puts"))] for side in sides}
    for epoch in range(1, final_epoch + 1):
        report_path = output / "epoch_validation" / f"epoch_{epoch:04d}" / "results.json"
        reports = json.loads(report_path.read_text())
        assert len(reports) == len(expected)
        assert {(r["year"], r["asset_class"], r["side"]) for r in reports} == expected
    print("Completed-run artifact and coverage checks passed.")


1T: /home/jlee153232/PycharmProjects/quant-orchestrator/artifacts/multirate_recovery/1T/warehouse_stream_20260912T201813Z
Log: /home/jlee153232/PycharmProjects/quant-orchestrator/artifacts/multirate_recovery/1T/warehouse_stream_20260912T201813Z.log
Coverage: /home/jlee153232/PycharmProjects/quant-orchestrator/artifacts/multirate_recovery/1T/warehouse_stream_20260912T201813Z/option_coverage.json
Failure details: none recorded
Completed-run artifact and coverage checks passed.

100B: /home/jlee153232/PycharmProjects/quant-orchestrator/artifacts/multirate_recovery/100B/warehouse_stream_20260913T032457Z
Log: /home/jlee153232/PycharmProjects/quant-orchestrator/artifacts/multirate_recovery/100B/warehouse_stream_20260913T032457Z.log
Coverage: /home/jlee153232/PycharmProjects/quant-orchestrator/artifacts/multirate_recovery/100B/warehouse_stream_20260913T032457Z/option_coverage.json
Failure details: /home/jlee153232/PycharmProjects/quant-orchestrator/artifacts/multirate_recovery/100B/warehouse

### Implementation references

- [Warehouse documents and coverage](../quant_orchestrator/research_tools/warehouse_multirate.py)
- [Shared training objectives](../quant_orchestrator/research_tools/multirate_training_step.py)
- [Streaming training and epoch evaluation](../quant_orchestrator/research_tools/warehouse_multirate_training.py)
- [Frozen baskets and split adjustments](../quant_orchestrator/research_tools/frozen_option_adjustments.py)
- [Option backtest](../quant_orchestrator/platforms/backtesting_frameworks/frozen_option_backtest.py)
- [Workflow documentation](../docs/multirate-warehouse-streaming.md)

The saved outputs are an executed artifact review, not a new training run performed by this notebook.